# 08.02_Hub_genes_R

核心基因网络与演化轨迹。

- 当前文件：`analysis/08_hub_genes/08.02_Hub_genes_R.ipynb`
- 原始来源：`Codes/08.02_R_Hub_genes.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`DNBr`, `Matrix`, `RColorBrewer`, `Seurat`, `dplyr`, `ggplot2`, `igraph`, `matrixStats`, `patchwork`, `pheatmap`, `reshape2`, `stringr`, `tidyr`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


### Network

In [ ]:
library(Seurat)
library(dplyr)
library(reshape2)
library(matrixStats)
library(DNBr)
library(ggplot2)

In [ ]:
# 读取您保存的 RDS 文件
out_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/DNB_analysis"
neural_working <- readRDS(paste0(out_dir, "/neural_working.rds"))
neural_working

In [ ]:
# 检查数据是否加载成功以及分组情况
table(neural_working$species, neural_working$Clade)

In [ ]:
table(neural_working$species, neural_working$Phylum)

In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
})

# =========================
# 用户可调参数
# =========================
assay_use   <- "integrated"
layer_use   <- "data"       # Seurat v5 用 layer；v4 会走 slot
slot_use    <- "data"       # Seurat v4 备用
cor_method  <- "pearson"    # "pearson" 或 "spearman"
edge_thr    <- 0.1         # 相关性阈值（建议用 abs 阈值）
use_abs     <- TRUE         # TRUE: |cor| >= thr；FALSE: cor >= thr
# out_dir     <- "species_networks"
out_dir     <- paste0("/share/home/zhangze/zz/NeuralOrigin/Data/08.HubGenes/Networks/", cor_method, edge_thr)
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# =========================
# 1) 提取整合矩阵 + 按 species 分组
# =========================

# 兼容 Seurat v4/v5 的 GetAssayData 调用
get_expr_mat <- function(seu, assay = "integrated", layer = "data", slot = "data") {
  # Seurat v5: GetAssayData(..., layer=)
  mat <- tryCatch(
    GetAssayData(seu, assay = assay, layer = layer),
    error = function(e) {
      # Seurat v4: GetAssayData(..., slot=)
      GetAssayData(seu, assay = assay, slot = slot)
    }
  )
  return(mat)
}

expr_all <- get_expr_mat(neural_working, assay = assay_use, layer = layer_use, slot = slot_use)
# expr_all: features(OGs) x cells

if (!"species" %in% colnames(neural_working@meta.data)) {
  stop("neural_working@meta.data 中没有 species 列，请检查列名。")
}

species_vec <- neural_working$species
species_levels <- sort(unique(as.character(species_vec)))

message("Species levels: ", paste(species_levels, collapse = ", "))
message("Expression matrix: ", nrow(expr_all), " OGs x ", ncol(expr_all), " cells")

# =========================
# 2) 每组：过滤全零OGs + 计算相关性矩阵
# =========================

# 计算相关性并剪枝导出边表
build_and_export_network <- function(expr_mat, species_name,
                                     method = "pearson",
                                     thr = 0.3,
                                     use_abs = TRUE,
                                     out_dir = ".") {
  # expr_mat: OG x cell （可以是 dgCMatrix）
  if (ncol(expr_mat) < 3) {
    warning(sprintf("[%s] cells < 3, 跳过（相关性不稳定）", species_name))
    return(invisible(NULL))
  }

  # (1) 过滤全零OG（在该 species 子矩阵中全为0）
  # Matrix::rowSums 对稀疏矩阵很快
  rs <- Matrix::rowSums(expr_mat != 0)
  keep_genes <- rs > 0
  expr_mat2 <- expr_mat[keep_genes, , drop = FALSE]

  message(sprintf("[%s] kept OGs: %d / %d ; cells: %d",
                  species_name, nrow(expr_mat2), nrow(expr_mat), ncol(expr_mat2)))

  if (nrow(expr_mat2) < 2) {
    warning(sprintf("[%s] kept OGs < 2, 跳过", species_name))
    return(invisible(NULL))
  }

  # (2) 计算基因-基因相关性：对基因向量在细胞上做相关
  # cor() 需要 dense matrix；2216xN 通常可接受（每组最多3000 cells）
  X <- as.matrix(expr_mat2)          # OG x cell
  C <- suppressWarnings(cor(t(X), method = method, use = "pairwise.complete.obs"))
  diag(C) <- 0

  # (3) 阈值剪枝：保留边
  if (use_abs) {
    sel <- which(abs(C) >= thr, arr.ind = TRUE)
  } else {
    sel <- which(C >= thr, arr.ind = TRUE)
  }

  # 只保留上三角，避免重复边（i<j）
  sel <- sel[sel[,1] < sel[,2], , drop = FALSE]
  if (nrow(sel) == 0) {
    warning(sprintf("[%s] 无边满足阈值 thr=%.3f (use_abs=%s)", species_name, thr, use_abs))
    # 仍然导出空边表 + 统计
    edges <- data.frame(source=character(), target=character(), weight=numeric(),
                        stringsAsFactors = FALSE)
  } else {
    genes <- rownames(C)
    edges <- data.frame(
      source = genes[sel[,1]],
      target = genes[sel[,2]],
      weight = C[sel],
      stringsAsFactors = FALSE
    )
  }

  # (4) 导出：边表 + 简单统计
  edge_file <- file.path(out_dir, sprintf("%s_%s_thr%s_edges.tsv",
                                         species_name, method, format(thr, scientific = FALSE)))
  write.table(edges, file = edge_file, sep = "\t", quote = FALSE,
              row.names = FALSE, col.names = TRUE)

  stat <- data.frame(
    species = species_name,
    method  = method,
    thr     = thr,
    use_abs = use_abs,
    n_cells = ncol(expr_mat2),
    n_ogs   = nrow(expr_mat2),
    n_edges = nrow(edges),
    stringsAsFactors = FALSE
  )
  stat_file <- file.path(out_dir, sprintf("%s_%s_thr%s_stats.tsv",
                                         species_name, method, format(thr, scientific = FALSE)))
  write.table(stat, file = stat_file, sep = "\t", quote = FALSE,
              row.names = FALSE, col.names = TRUE)

  message(sprintf("[%s] edges exported: %d -> %s", species_name, nrow(edges), edge_file))
  return(invisible(list(edges = edges, stats = stat)))
}

# =========================
# 3) 循环9个 species，得到9个子网络并导出
# =========================
all_stats <- list()

for (sp in species_levels) {
  cell_ids <- colnames(neural_working)[which(as.character(species_vec) == sp)]
  if (length(cell_ids) == 0) next

  expr_sp <- expr_all[, cell_ids, drop = FALSE]
  res <- build_and_export_network(expr_sp, species_name = sp,
                                  method = cor_method,
                                  thr = edge_thr,
                                  use_abs = use_abs,
                                  out_dir = out_dir)
  if (!is.null(res)) all_stats[[sp]] <- res$stats
}

# 汇总统计
if (length(all_stats) > 0) {
  stats_df <- do.call(rbind, all_stats)
  write.table(stats_df, file = file.path(out_dir, "ALL_species_network_stats.tsv"),
              sep = "\t", quote = FALSE, row.names = FALSE)
  print(stats_df)
}

message("Done. Output dir: ", normalizePath(out_dir))

In [ ]:
suppressPackageStartupMessages({
  library(igraph)
  library(dplyr)
  library(tidyr)
  library(stringr)
})

# =========================
# 0) Inputs
# =========================
core_OGs <- c("OG0000203","OG0000133","OG0000036","OG0000260","OG0000112","OG0000166")

species_orders <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
species_use <- rev(species_orders)  # 反转顺序

# 你之前导出的网络边表目录
out_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/08.HubGenes/Networks/pearson0.1"

# 与之前一致：文件名形如  Dare_pearson_thr0.3_edges.tsv
cor_method <- "pearson"  # 或 "spearman"
edge_thr   <- 0.1

# 输出
out_file <- file.path(out_dir, sprintf("coreOG_degree_entropy_%s_thr%s.tsv",
                                       cor_method, format(edge_thr, scientific = FALSE)))

# =========================
# 1) Helper: load graph for one species
# =========================
load_species_graph <- function(species, out_dir, method, thr) {
  f <- file.path(out_dir, sprintf("%s_%s_thr%s_edges.tsv",
                                  species, method, format(thr, scientific = FALSE)))
  if (!file.exists(f)) {
    warning(sprintf("[%s] edge file not found: %s", species, f))
    return(NULL)
  }
  edges <- read.delim(f, stringsAsFactors = FALSE)

  # 允许空边表
  if (nrow(edges) == 0) {
    # 建一个空图（无边）；顶点信息未知，所以返回空图
    return(make_empty_graph(directed = FALSE))
  }

  g <- graph_from_data_frame(edges[, c("source", "target")], directed = FALSE)
  E(g)$weight <- edges$weight
  return(g)
}

# =========================
# 2) Helper: local entropy for a node
# =========================
local_entropy <- function(g, v) {
  if (!v %in% V(g)$name) return(NA_real_)
  d <- degree(g, v)
  if (is.na(d) || d == 0) return(0)

  nb <- neighbors(g, v)
  if (length(nb) == 0) return(0)

  k_nb <- degree(g, nb)
  s <- sum(k_nb)
  if (is.na(s) || s <= 0) return(0)

  p <- k_nb / s
  # 避免 log(0)
  p <- p[p > 0]
  if (length(p) == 0) return(0)

  H <- -sum(p * log(p))
  return(as.numeric(H))
}

# =========================
# 3) Main loop: species x coreOG
# =========================
res_list <- list()

for (sp in species_use) {
  g <- load_species_graph(sp, out_dir, cor_method, edge_thr)

  # 如果没找到文件，仍然输出占位行
  if (is.null(g)) {
    tmp <- data.frame(
      species = sp,
      OG = core_OGs,
      present_in_graph = FALSE,
      degree = NA_integer_,
      n_neighbors = NA_integer_,
      local_entropy = NA_real_,
      stringsAsFactors = FALSE
    )
    res_list[[sp]] <- tmp
    next
  }

  # 若图为空且无顶点名，则所有 coreOG 都视为不存在
  has_names <- length(V(g)) > 0 && !is.null(V(g)$name) && any(nzchar(V(g)$name))

  tmp <- lapply(core_OGs, function(og) {
    present <- has_names && (og %in% V(g)$name)
    if (!present) {
      return(data.frame(
        species = sp, OG = og, present_in_graph = FALSE,
        degree = NA_integer_, n_neighbors = NA_integer_, local_entropy = NA_real_,
        stringsAsFactors = FALSE
      ))
    } else {
      d <- degree(g, og)
      nb <- neighbors(g, og)
      nn <- length(nb)
      H <- local_entropy(g, og)

      return(data.frame(
        species = sp, OG = og, present_in_graph = TRUE,
        degree = as.integer(d),
        n_neighbors = as.integer(nn),
        local_entropy = as.numeric(H),
        stringsAsFactors = FALSE
      ))
    }
  }) %>% bind_rows()

  res_list[[sp]] <- tmp
}

res <- bind_rows(res_list)

# 按你要求的物种顺序（反转后的）与 core_OGs 顺序排序
res$species <- factor(res$species, levels = species_use)
res$OG <- factor(res$OG, levels = core_OGs)

res <- res %>%
  arrange(species, OG)

# =========================
# 4) Export
# =========================
write.table(res, file = out_file, sep = "\t", quote = FALSE,
            row.names = FALSE, col.names = TRUE)

print(res)
message("Exported: ", normalizePath(out_file))

### Plot

In [ ]:
library(ggplot2)
library(tidyr)
library(pheatmap)

In [ ]:
# file_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/08.HubGenes/Networks/pearson0.05/coreOG_degree_entropy_pearson_thr0.05.tsv"
file_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/08.HubGenes/Networks/pearson0.1/coreOG_degree_entropy_pearson_thr0.1.tsv"
# file_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/08.HubGenes/Networks/pearson0.3/coreOG_degree_entropy_pearson_thr0.3.tsv"
# file_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/08.HubGenes/Networks/pearson0.5/coreOG_degree_entropy_pearson_thr0.5.tsv"
res_df <- read.table(file_path, header = TRUE, sep = "\t", stringsAsFactors = FALSE)
res_df

In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(RColorBrewer)

# 定义演化顺序（根据您的物种列表）
species_order <- c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Auco", "Clhe", "Neve", "Dare")

# 映射门类（Phylum）
phylum_map <- c(
  "Spla" = "Porifera",
  "ClH23" = "Placozoa", "HoH13" = "Placozoa", "TrH2" = "Placozoa", "TrH1" = "Placozoa",
  "Auco" = "Cnidaria", "Clhe" = "Cnidaria", "Neve" = "Cnidaria",
  "Dare" = "Bilateria"
)

# 数据预处理
res_df_plot <- res_df %>%
  mutate(species = factor(species, levels = species_order)) %>%
  mutate(phylum = phylum_map[as.character(species)]) %>%
  mutate(phylum = factor(phylum, levels = c("Porifera", "Placozoa", "Cnidaria", "Bilateria")))

In [ ]:
# 绘图 A: Degree 演化
p1 <- ggplot(res_df_plot, aes(x = species, y = degree, group = OG, color = OG)) +
  geom_line(size = 1, alpha = 0.7) +
  geom_point(size = 3) +
  facet_wrap(~phylum, scales = "free_x", nrow = 1) +
  theme_bw() +
  scale_color_brewer(palette = "Set1") +
  labs(title = "Network Degree Evolution (Pearson 0.3)", y = "Degree (No. of Strong Neighbors)", x = "") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

# 绘图 B: Entropy 演化
p2 <- ggplot(res_df_plot, aes(x = species, y = local_entropy, group = OG, color = OG)) +
  geom_line(size = 1, alpha = 0.7) +
  geom_point(size = 3) +
  facet_wrap(~phylum, scales = "free_x", nrow = 1) +
  theme_bw() +
  scale_color_brewer(palette = "Set1") +
  labs(title = "Local Entropy Evolution (Pearson 0.3)", y = "Local Shannon Entropy", x = "Evolutionary Trajectory") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

# 合并显示
p1 / p2 + plot_layout(guides = "collect")

In [ ]:
ggplot(res_df_plot %>% filter(!is.na(degree)), aes(x = degree, y = local_entropy, color = phylum)) +
  geom_point(aes(shape = OG), size = 4, alpha = 0.8) +
  geom_text(aes(label = species), vjust = -1, size = 3, check_overlap = TRUE) +
  theme_minimal() +
  scale_color_manual(values = c("Porifera" = "#fba414", "Placozoa" = "#EC2B24", "Cnidaria" = "#2A52BE", "Bilateria" = "#43b244")) +
  labs(title = "Network State Phase Space",
       subtitle = "High Degree & High Entropy indicates Critical Transitions",
       x = "Degree (Connectivity)", y = "Local Entropy (Complexity)") +
  theme(legend.position = "right")

In [ ]:
library(pheatmap)
library(tidyr)
library(dplyr)

# 准备热图矩阵 (使用基础 R 语法替代 column_to_rownames)
entropy_df_wide <- res_df_plot %>%
  select(species, OG, local_entropy) %>%
  spread(species, local_entropy)

# 将第一列设置为行名，然后删除第一列
entropy_mat <- as.matrix(entropy_df_wide[, -1])
rownames(entropy_mat) <- entropy_df_wide$OG

# 处理 NA 值为 0（代表在 0.3 阈值下没有显著连接）
entropy_mat[is.na(entropy_mat)] <- 0

# 绘图
pheatmap(entropy_mat, 
         cluster_cols = FALSE, 
         cluster_rows = TRUE,
         color = colorRampPalette(c("#f0f0f0", "#feb24c", "#e31a1c"))(100),
         main = "Local Entropy Map",
         display_numbers = TRUE,
         number_color = "black",
         fontsize_number = 8)

In [ ]:
library(ggplot2)
library(dplyr)
library(patchwork)

# 1. 显式定义 9 个物种的演化顺序
species_order <- c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Auco", "Clhe", "Neve", "Dare")

# 2. 预处理数据：确保因子顺序，并处理缺失值
res_df_plot <- res_df %>%
  filter(!is.na(OG)) %>% # 确保 OG 存在
  mutate(species = factor(species, levels = species_order)) %>%
  # 将 NA 填充为 0 这样连线才不会中断（如果该 OG 在该物种中没边，则度为0，熵为0）
  mutate(degree = ifelse(is.na(degree), 0, degree),
         local_entropy = ifelse(is.na(local_entropy), 0, local_entropy))

# 3. 绘图 A: Degree 连续演化轨迹
p1 <- ggplot(res_df_plot, aes(x = species, y = degree, group = OG, color = OG)) +
  # 添加一条垂直线区分 Non-neural 和 Neural (可选，在 TrH1 和 Auco 之间)
  geom_vline(xintercept = 5.5, linetype = "dashed", color = "grey70") +
  geom_line(linewidth = 1, alpha = 0.8) +
  geom_point(size = 3) +
  theme_classic() + # 使用更简洁的经典主题
  scale_color_brewer(palette = "Set1") +
  labs(title = "Network Degree Connectivity Trajectory", 
       y = "Degree (No. of Neighbors)", x = "") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, face = "bold"),
        legend.position = "none")

# 4. 绘图 B: Entropy 连续演化轨迹
p2 <- ggplot(res_df_plot, aes(x = species, y = local_entropy, group = OG, color = OG)) +
  geom_vline(xintercept = 5.5, linetype = "dashed", color = "grey70") +
  geom_line(linewidth = 1, alpha = 0.8) +
  geom_point(size = 3) +
  theme_classic() +
  scale_color_brewer(palette = "Set1") +
  labs(title = "Local Entropy Complexity Trajectory", 
       y = "Local Shannon Entropy", x = "Species (Ordered by Evolution)") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, face = "bold"))

# 5. 合并显示
p1 / p2 + plot_layout(guides = "collect") & 
  theme(legend.position = "right")

绘制图片

In [ ]:
fig_dir <- '/share/home/zhangze/zz/NeuralOrigin/Figures'

In [ ]:
library(ggplot2)
library(dplyr)
library(patchwork)

# 1. 显式定义 9 个物种的演化顺序
species_order <- c("Spla", "ClH23", "HoH13", "TrH2", "TrH1", "Auco", "Clhe", "Neve", "Dare")

# 2. 自定义颜色 (在此处修改颜色代码)
# 建议为每个 OG 指定固定颜色，确保在不同图表中保持一致
my_colors <- c(
  "OG0000203" = "#E41A1C", # 红色
  "OG0000133" = "#377EB8", # 蓝色
  "OG0000036" = "#4DAF4A", # 绿色
  "OG0000260" = "#984EA3", # 紫色
  "OG0000112" = "#FF7F00", # 橙色
  "OG0000166" = "#A65628"  # 棕色
)

# 3. 预处理数据
res_df_plot <- res_df %>%
  filter(!is.na(OG)) %>%
  mutate(species = factor(species, levels = species_order)) %>%
  # 处理缺失值以保证折线连续
  mutate(degree = ifelse(is.na(degree), 0, degree),
         local_entropy = ifelse(is.na(local_entropy), 0, local_entropy))

# 4. 绘图 A: Degree 演化轨迹
p1 <- ggplot(res_df_plot, aes(x = species, y = degree, group = OG, color = OG)) +
  geom_vline(xintercept = 5.5, linetype = "dashed", color = "grey70") + # 演化分界线
  geom_line(linewidth = 1, alpha = 0.8) +
  geom_point(size = 3) +
  scale_color_manual(values = my_colors) + # 使用自定义颜色
  theme_classic(base_size = 14) +
  labs(title = "Network Degree Connectivity Trajectory", 
       y = "Degree (No. of Neighbors)", x = "") +
  theme(axis.text.x = element_blank(), # p1 不显示坐标轴文字，节省空间
        axis.ticks.x = element_blank(),
        legend.position = "none")

# 5. 绘图 B: Entropy 演化轨迹
p2 <- ggplot(res_df_plot, aes(x = species, y = local_entropy, group = OG, color = OG)) +
  geom_vline(xintercept = 5.5, linetype = "dashed", color = "grey70") +
  geom_line(linewidth = 1, alpha = 0.8) +
  geom_point(size = 3) +
  scale_color_manual(values = my_colors) + # 使用自定义颜色
  theme_classic(base_size = 14) +
  labs(title = "Local Entropy Complexity Trajectory", 
       y = "Local Shannon Entropy", x = "Species (Ordered by Evolution)") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, face = "bold"))

# 6. 合并图像
final_plot <- p1 / p2 + 
  plot_layout(guides = "collect") & 
  theme(legend.position = "right")

# 7. 打印预览
print(final_plot)

# 8. 导出 PDF (指定 300 DPI 质量，虽然 PDF 是矢量图，但设置尺寸很重要)
# 文件名可根据您的路径修改
out_file <- "/80.CoreOG_Evolution_Trajectory_Pearson0.1.pdf"

ggsave(paste0(fig_dir, out_file), 
       plot = final_plot, 
       device = "pdf", 
       width = 10, 
       height = 8, 
       units = "in", 
       dpi = 300)

message("可视化分析已完成并保存至: ", out_file)